# 01 - Download e Preparação do Dataset ArtiFact (Faces)

Download do dataset ArtiFact (Kaggle) e extração apenas das imagens de **rostos humanos** para uso como conjunto de validação cross-generator.

O ArtiFact contém imagens de múltiplos geradores (StyleGAN, ProGAN, Stable Diffusion, DALL-E, entre outros) e múltiplas categorias. Aqui filtramos apenas a categoria de faces humanas.

> Requer Kaggle API configurada (`~/.kaggle/access_token`).

In [ ]:
import subprocess
import sys
import zipfile
import shutil
import random
from pathlib import Path
from tqdm import tqdm

In [ ]:
PROJECT_ROOT    = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT       = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"

ARTIFACT_RAW  = DATA_ROOT / "raw" / "artifact"
ARTIFACT_FACES = DATA_ROOT / "raw" / "artifact_faces"
KAGGLE_DATASET = "awsaf49/artifact-dataset"

ARTIFACT_RAW.mkdir(parents=True, exist_ok=True)

print("Download em:", ARTIFACT_RAW)
print("Faces filtradas em:", ARTIFACT_FACES)

## 1. Download

In [ ]:
import time as _time

zip_path = ARTIFACT_RAW / "artifact-dataset.zip"

# Dataset grande (~31 GB). Se o download pela API estiver muito lento,
# baixe manualmente em https://www.kaggle.com/datasets/awsaf49/artifact-dataset
# e coloque o artifact-dataset.zip em ARTIFACT_RAW.
if any(ARTIFACT_RAW.iterdir()):
    print("Arquivos já existem em:", ARTIFACT_RAW)
else:
    print("Baixando ArtiFact (~31 GB, pode levar horas)...")
    proc = subprocess.Popen(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", KAGGLE_DATASET, "-p", str(ARTIFACT_RAW)],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    # progresso por crescimento do arquivo (total desconhecido até o fim)
    pbar = tqdm(unit="B", unit_scale=True, unit_divisor=1024, desc="Download")
    last = 0
    while proc.poll() is None:
        cur = sum(f.stat().st_size for f in ARTIFACT_RAW.glob("*") if f.is_file())
        pbar.update(cur - last)
        last = cur
        _time.sleep(2)
    pbar.close()
    if proc.wait() != 0:
        raise RuntimeError("Falha no download do Kaggle. Verifique o token ou baixe manualmente.")
    print("Download concluído.")

## 2. Exploração da Estrutura

Lista todas as fontes (pastas de topo) e marca quais contêm imagens de face. Serve para verificar que o filtro da seção 3 cobre todas as fontes de face e para conferir os nomes exatos (ex: `metfaces`, famílias `stylegan`).

In [ ]:
zips = sorted(ARTIFACT_RAW.glob("*.zip"))
print("Arquivos zip encontrados:")
for z in zips:
    print(f"  {z.name} ({z.stat().st_size / 1e9:.1f} GB)")

In [ ]:
from collections import Counter

FACE_KW = ["ffhq", "celebahq", "celeba", "face"]
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")

with zipfile.ZipFile(zips[0], "r") as zf:
    all_names = zf.namelist()

# conta imagens por fonte (topo) e quantas são de face
total_by_src = Counter()
face_by_src  = Counter()
for n in all_names:
    if not n.lower().endswith(IMG_EXT):
        continue
    src = n.split("/")[0]
    total_by_src[src] += 1
    if any(kw in n.lower() for kw in FACE_KW):
        face_by_src[src] += 1

print(f"Total de imagens: {sum(total_by_src.values())}")
print(f"Fontes (top-level): {len(total_by_src)}\n")
print(f"{'fonte':<28}{'imagens':>10}{'faces':>10}")
print("-" * 48)
for src in sorted(total_by_src, key=lambda s: -total_by_src[s]):
    marca = "  <- face" if face_by_src[src] else ""
    print(f"{src:<28}{total_by_src[src]:>10}{face_by_src[src]:>10}{marca}")

## 3. Filtro de Faces e Rótulo

O ArtiFact organiza por **fonte** no topo do caminho. O rótulo vem da fonte:

- **Reais** (face humana): `ffhq`, `celebahq`, `metfaces`
- **Falsas**: qualquer outra fonte (gerador) cujas imagens sejam de face — identificadas por subpasta com `ffhq`/`celeba`/`face` no caminho (ex: `cips/cips-ffhq`, `diffusion_gan/ffhq-data`, `gansformer/ffhq_images`, `face_synthetics`)

`afhq` (face de **animal**) é naturalmente excluído por não casar com os keywords de face humana.

Cada imagem extraída guarda a fonte no nome (`fonte__arquivo.jpg`) e num `manifest.csv` — isso evita colisão de nomes e permite, na análise, excluir geradores específicos (ex: a própria família StyleGAN, para um teste puramente cross-generator).

In [ ]:
from collections import Counter

# ── configuração ──────────────────────────────────────────────
REAL_FACE_SOURCES = {"ffhq", "celebahq", "metfaces"}        # face humana real
FACE_KEYWORDS     = ["ffhq", "celebahq", "celeba", "face"]  # caminho indica face humana
# Exclui a família StyleGAN: o treino (140k) JÁ é StyleGAN, então incluí-la aqui
# descaracterizaria o teste cross-generator. (sfhq, derivado de StyleGAN2, já fica
# de fora naturalmente porque sua subpasta não casa com os keywords de face.)
EXCLUDE_SOURCES   = {"stylegan1", "stylegan2", "stylegan3"}

# caps por fonte. Há só 3 fontes reais vs 8 geradores, então as reais recebem
# um cap maior para equilibrar os totais (real ~15.3k vs fake ~16k).
FAKE_CAP_PER_SOURCE = 2000    # 8 geradores * 2000 = 16000 fake
REAL_CAP_PER_SOURCE = 7000    # ffhq 7000 + celebahq 7000 + metfaces 1336 = 15336 real

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")
# ──────────────────────────────────────────────────────────────

def top_source(name):
    return name.split("/")[0]

def is_face_path(name):
    p = name.lower()
    return any(kw in p for kw in FACE_KEYWORDS)

def classify(name):
    src = top_source(name)
    if src in EXCLUDE_SOURCES:
        return None
    return "real" if src in REAL_FACE_SOURCES else "fake"

def cap_for(src):
    return REAL_CAP_PER_SOURCE if src in REAL_FACE_SOURCES else FAKE_CAP_PER_SOURCE

with zipfile.ZipFile(zips[0], "r") as zf:
    all_names = zf.namelist()

face_files = [n for n in all_names if is_face_path(n) and n.lower().endswith(IMG_EXTS)]

# contagem por fonte e rótulo (refletindo as exclusões)
by_source = Counter(top_source(n) for n in face_files)
print(f"Arquivos de face encontrados: {len(face_files)}\n")
print(f"{'fonte':<25}{'qtd':>8}  rótulo")
print("-" * 45)
for src, c in sorted(by_source.items(), key=lambda x: -x[1]):
    if src in EXCLUDE_SOURCES:
        lab = "EXCLUÍDA"
    elif src in REAL_FACE_SOURCES:
        lab = "real"
    else:
        lab = "fake"
    print(f"{src:<25}{c:>8}  {lab}")

## 4. Extração para Pasta Limpa (real / fake)

In [ ]:
import pandas as pd

(ARTIFACT_FACES / "real").mkdir(parents=True, exist_ok=True)
(ARTIFACT_FACES / "fake").mkdir(parents=True, exist_ok=True)

# embaralha para amostrar diverso dentro de cada fonte (cap por fonte)
shuffled = face_files[:]
random.Random(42).shuffle(shuffled)

per_source = Counter()
manifest = []

with zipfile.ZipFile(zips[0], "r") as zf:
    for name in tqdm(shuffled, desc="Extraindo faces"):
        label = classify(name)
        if label is None:
            continue
        src = top_source(name)
        if per_source[src] >= cap_for(src):
            continue
        safe_name = f"{src}__{Path(name).name}"   # evita colisão entre fontes
        dst = ARTIFACT_FACES / label / safe_name
        if not dst.exists():
            dst.write_bytes(zf.read(name))
        per_source[src] += 1
        manifest.append({"file": safe_name, "label": label, "source": src})

mdf = pd.DataFrame(manifest)
mdf.to_csv(ARTIFACT_FACES / "manifest.csv", index=False)

print("Por fonte:")
print(mdf.groupby(["label", "source"]).size().to_string())
print(f"\nTotal — real: {(mdf.label=='real').sum()} | fake: {(mdf.label=='fake').sum()}")
print("Manifest salvo em:", ARTIFACT_FACES / "manifest.csv")

## 5. Verificação

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def list_imgs(folder):
    files = []
    for e in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        files += list(folder.glob(e))
    return files

real_imgs = list_imgs(ARTIFACT_FACES / "real")
fake_imgs = list_imgs(ARTIFACT_FACES / "fake")

print(f"Reais:  {len(real_imgs)}")
print(f"Falsas: {len(fake_imgs)}")

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for col, p in enumerate(random.sample(real_imgs, min(6, len(real_imgs)))):
    axes[0, col].imshow(Image.open(p).convert("RGB").resize((128, 128)))
    axes[0, col].set_title(p.name.split("__")[0], fontsize=7)
    axes[0, col].axis("off")
for col, p in enumerate(random.sample(fake_imgs, min(6, len(fake_imgs)))):
    axes[1, col].imshow(Image.open(p).convert("RGB").resize((128, 128)))
    axes[1, col].set_title(p.name.split("__")[0], fontsize=7)
    axes[1, col].axis("off")
axes[0, 0].set_ylabel("real")
axes[1, 0].set_ylabel("fake")
plt.suptitle("ArtiFact — Amostras de Faces (título = fonte)")
plt.tight_layout()
plt.show()